In [1]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import gc
import json
import re
import numpy as np
from tqdm import tqdm
import evaluate
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

print("CUDA Available:", torch.cuda.is_available())

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
C:\Users\Legion\AppData\Local\Temp\ipykernel_10844\4251492608.py:11: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA Available: True


In [2]:
BASE_MODEL_DIR = "base_llama3_model"
ADAPTER_DIR = "adapters/FUNSD_Adapter"
TEST_FILE = "phase1_funsd_test.jsonl" 

In [3]:
print("\nLoading models for FUNSD inference...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_DIR, 
    max_seq_length = 4096, 
    dtype = None,
    load_in_4bit = True,
)

print(f"Loading FUNSD LoRA Adapter from: {ADAPTER_DIR}...")
model.load_adapter(ADAPTER_DIR)
FastLanguageModel.for_inference(model)


Loading models for FUNSD inference...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading FUNSD LoRA Adapter from: adapters/FUNSD_Adapter...


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3072, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=3072, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=307

In [4]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an intelligent AI form extraction agent. Analyze the raw OCR text from the scanned document.
CRITICAL RULES:
1. Extract ALL key-value pairs found in the form.
2. Output a brief 'Thought:' section summarizing the document type and its semantic layout. Do NOT use or mention bounding box coordinates.
3. Output the final result as a valid, parsable JSON object where the extracted labels are the keys, and the extracted data are the values.
4. Do NOT invent keys or values not present in the text.

Format your output EXACTLY like this:
Thought: [Briefly describe what the document is and how the information is structured logically]
JSON: 
{{
  "Extracted Key 1": "Extracted Value 1",
  "Extracted Key 2": "Extracted Value 2"
}}

### Input:
{}

### Response:
{}"""

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [5]:
print(f"\nLoading test data from {TEST_FILE}...")
test_samples = []
if os.path.exists(TEST_FILE):
    with open(TEST_FILE, "r", encoding="utf-8") as f:
        for line in f:
            test_samples.append(json.loads(line))
else:
    print(f"Warning: {TEST_FILE} not found. Only the custom form test will run.")


Loading test data from phase1_funsd_test.jsonl...


In [6]:
def parse_and_flatten_json(text):
    """Extracts JSON and flattens it into a semantic string for BERTScore"""
    try:
        if "JSON:" in text:
            json_block = text.split("JSON:")[1].strip()
        else:
            json_block = text.strip()
            
        json_block = re.sub(r"```json\s*", "", json_block)
        json_block = re.sub(r"```\s*$", "", json_block)
        
        parsed_dict = json.loads(json_block)
        
        flattened = " | ".join([f"{str(k)}: {str(v)}" for k, v in parsed_dict.items()])
        return flattened if flattened else "[EMPTY_FORM]"
        
    except Exception:
        return "[JSON_PARSING_ERROR]"

In [7]:
pred_strings = []
ref_strings = []

if test_samples:
    print("\nAgent is evaluating unseen forms...")
    for sample in tqdm(test_samples, desc="Evaluating Forms"): 
        unseen_form = sample["input"]
        gold_full_text = sample["output"] 
        
        inputs = tokenizer([alpaca_prompt.format(unseen_form, "Thought: ")], return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            **inputs, 
            max_new_tokens = 800,      
            temperature = 0.1,          
            top_p = 0.9,                
            use_cache = True,
            repetition_penalty = 1.1,
            eos_token_id = terminators, 
        )
        
        generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        try:
            ai_full_text = generated_text.split("### Response:\n")[1].strip()
        except IndexError:
            ai_full_text = generated_text.strip()
            
        pred_strings.append(parse_and_flatten_json(ai_full_text))
        ref_strings.append(parse_and_flatten_json(gold_full_text))

    print("\nLoading BERTScore Model...")
    bertscore = evaluate.load("bertscore")

    scores = bertscore.compute(predictions=pred_strings, references=ref_strings, lang="en")
    overall_f1 = np.mean(scores['f1']) * 100

    print("\n" + "="*50)
    print("FUNSD EXTRACTION ACCURACY (BERTScore F1)")
    print("="*50)
    print(f"OVERALL DYNAMIC FORM SCORE: {overall_f1:.2f}%")
    print("="*50)


Agent is evaluating unseen forms...


Evaluating Forms: 100%|██████████| 47/47 [20:19<00:00, 25.94s/it]



Loading BERTScore Model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



FUNSD EXTRACTION ACCURACY (BERTScore F1)
OVERALL DYNAMIC FORM SCORE: 85.41%


In [8]:
def evaluate_custom_form(ocr_text, title):
    print(f"\nEvaluating: {title}...\n" + "="*50)
    
    inputs = tokenizer([alpaca_prompt.format(ocr_text, "Thought: ")], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 800,       
        temperature = 0.1,          
        top_p = 0.9,                
        use_cache = True,
        repetition_penalty = 1.1,   
        eos_token_id = terminators, 
    )

    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    try:
        response_only = generated_text.split("### Response:\n")[1].strip()
        
        json_start = response_only.find('{')
        json_end = response_only.rfind('}') + 1 
        
        if json_start != -1 and json_end != 0:
            thoughts = response_only[:json_start].strip()
            clean_json = response_only[json_start:json_end]
            
            print(thoughts)
            print("\nJSON:")
            print(clean_json)
        else:
            print(response_only) 
            
    except IndexError:
        print(generated_text.strip())
    print("="*50)

In [9]:
# Unseen Custom Form
unseen_patient_form = "PATIENT REGISTRATION FORM [50, 20, 300, 40] || Patient Name: [50, 60, 150, 80] || Sarah Jenkins [160, 60, 280, 80] || Date of Birth: [50, 90, 160, 110] || 12/04/1985 [170, 90, 260, 110] || Insurance Provider: [50, 120, 200, 140] || Blue Cross Blue Shield [210, 120, 400, 140] || Emergency Contact: [50, 150, 200, 170] || Michael Jenkins [210, 150, 350, 170] || Phone Number: [50, 180, 160, 200] || (555) 019-8372 [170, 180, 290, 200] || Signature: [50, 250, 130, 270] || S. Jenkins [140, 250, 230, 270]"

evaluate_custom_form(unseen_patient_form, "Patient Registration Form")


Evaluating: Patient Registration Form...
Thought:  This document is a patient registration form, which includes essential personal details about the individual seeking medical care. The top section identifies the form's purpose, while the middle part collects specific demographic information such as name, date of birth, insurance provider, and emergency contact details. The bottom section requires a signature to validate the form. Each labeled field corresponds directly to the extracted value, providing a clear and organized structure for capturing critical patient data.
JSON:

JSON:
{"Patient Name": "Sarah Jenkins", "Date of Birth": "12/04/1985", "Insurance Provider": "Blue Cross Blue Shield", "Emergency Contact": "Michael Jenkins", "Phone Number": "(555) 019-8372", "Signature": "S. Jenkins"}


In [10]:
# HARD SHUTDOWN & UNLOAD FROM GPU

import torch
import gc

print("Initiating VRAM Hard-Shutdown for Local Models...")

# 1. Track memory before cleanup
if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated Before: {vram_before:.2f} GB")

# 2. List of every heavy object that might be trapped in memory
heavy_objects = [
    'model', 'tokenizer', 'trainer', 'bertscore', 'rouge', 
    'dataset', 'train_dataset', 'eval_dataset', 'split_dataset',
    'inputs', 'outputs'
]

# 3. Delete them dynamically if they exist
for obj in heavy_objects:
    if obj in globals():
        print(f"Unloading '{obj}' from memory...")
        del globals()[obj]

# 4. Force aggressive Garbage Collection (Run twice to clear circular references)
gc.collect()
gc.collect()

# 5. Flush the GPU Cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # Clears memory shared between backend processes
    torch.cuda.synchronize()  # Waits for all GPU operations to completely finish
    
    # Track memory after cleanup
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated After:  {vram_after:.2f} GB")

print("\nGPU Memory Cleared. Your VRAM is now completely empty!")

Initiating VRAM Hard-Shutdown for Local Models...
VRAM Allocated Before: 3.29 GB
Unloading 'model' from memory...
Unloading 'tokenizer' from memory...
Unloading 'bertscore' from memory...
Unloading 'inputs' from memory...
Unloading 'outputs' from memory...
VRAM Allocated After:  2.29 GB

GPU Memory Cleared. Your VRAM is now completely empty!
